### Setup

In [2]:
!pip install -r requirements.txt

  Using cached transformer_lens-2.16.1-py3-none-any.whl.metadata (12 kB)
  Using cached sae_lens-6.17.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached unbabel_comet-2.2.7-py3-none-any.whl.metadata (19 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached circuitsvis-1.43.3-py3-none-any.whl.metadata (983 bytes)
  Using cached umap_learn-0.5.9.post2-py3-none-any.whl.metadata (25 kB)
  Using cached matplotlib-3.10.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached langdetect-1.0.9-py3-none-any.whl
  Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
  Using cached fasttext-0.9.3-cp312-cp312-linux_x86_64.whl
  Using cached sentence_transformers-5.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached beartype-0.14.1-py3-none-any.whl.metadata (28 kB)
  Using cached better_abc-0.0.3-py3-none-any.whl.meta

In [3]:
import os
import numpy as np
import torch as t
from sae_lens import (
    SAE,
    HookedSAETransformer,
)
import gc
import pickle
from datasets import load_dataset

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
from huggingface_hub import notebook_login
device = "cuda"
t.set_grad_enabled(False)
notebook_login()

### Load Model

In [5]:
!pip install hf_transfer
model_ver = "2b"
if model_ver == "2b":
    total_layers = 26
else:
    total_layers = 42
gemma: HookedSAETransformer = HookedSAETransformer.from_pretrained(f"gemma-2-{model_ver}", device=device)
tokenizer = gemma.tokenizer

  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


### Load dataset

In [6]:
# Load the Shakespeare dataset
dataset = load_dataset("notaphoenix/shakespeare_dataset")

# Get the train split
data = dataset['training']

In [7]:
# Sample and prepare the data
np.random.seed(42)

# Separate Shakespeare (label=1) and modern (label=0) texts
all_original = [example['text'] for example in data if example['label'] == 1]
all_modern = [example['text'] for example in data if example['label'] == 0]

print(f"Total Shakespeare texts: {len(all_original)}")
print(f"Total modern texts: {len(all_modern)}")

# Sample 1000 from each (or max available)
num_samples = min(1000, len(all_original), len(all_modern))
print(f"\nSampling {num_samples} examples from each")

original_indices = np.random.choice(len(all_original), size=num_samples, replace=False)
modern_indices = np.random.choice(len(all_modern), size=num_samples, replace=False)

original_prompts = [all_original[i] for i in original_indices]
modern_prompts = [all_modern[i] for i in modern_indices]

# Check first examples
print(f"\nOriginal (Shakespeare):")
print(original_prompts[0])
print(f"\nModern English:")
print(modern_prompts[0])

Total Shakespeare texts: 1932
Total modern texts: 1927

Sampling 1000 examples from each

Original (Shakespeare):
Now heaven hath all ,


Modern English:
I wouldn't even pee without dancing a waltz .



### Retrieve SAE Activations

In [8]:
sae_release = f"gemma-scope-{model_ver}-pt-res-canonical"
sae_id = "layer_0/width_16k/canonical"

blobs_path = f"/root/.cache/huggingface/hub/models--google--gemma-scope-{model_ver}-pt-res/blobs"

# Create output directories
os.makedirs(f"./sae_acts_{model_ver}/shakespeare", exist_ok=True)

# SMALLER BATCH SIZE: Process 10 examples at a time
batch_size = 10
num_batches = 100  # 10 * 100 = 1000 total

for i in range(7,total_layers):
    sae = SAE.from_pretrained(
        release=sae_release,
        sae_id=f"layer_{i}/width_16k/canonical",
        device=str(device),
    )[0]  # Get just the SAE object
    
    for j in range(num_batches):
        start_idx = j * batch_size
        end_idx = (j + 1) * batch_size
        
        # Process original Shakespeare text
        _, cache = gemma.run_with_cache_with_saes(original_prompts[start_idx:end_idx], saes=[sae])
        sae_acts_post = cache[f"{sae.cfg.metadata.get('hook_name')}.hook_sae_acts_post"].cpu()
        t.save(sae_acts_post, f"./sae_acts_{model_ver}/shakespeare/original_layer{i}_idx{end_idx}.pt")

        cache.to("cpu")
        del cache, sae_acts_post
        t.cuda.empty_cache()
        gc.collect()
        
        # Clean up blobs
        if os.path.exists(blobs_path):
            for item in os.listdir(blobs_path):
                item_path = os.path.join(blobs_path, item)
                if os.path.isfile(item_path):
                    os.remove(item_path)
        
        # Process modern English text
        _, cache = gemma.run_with_cache_with_saes(modern_prompts[start_idx:end_idx], saes=[sae])
        sae_acts_post = cache[f"{sae.cfg.metadata.get('hook_name')}.hook_sae_acts_post"].cpu()
        t.save(sae_acts_post, f"./sae_acts_{model_ver}/shakespeare/modern_layer{i}_idx{end_idx}.pt")

        cache.to("cpu")
        del cache, sae_acts_post
        t.cuda.empty_cache()
        gc.collect()
        
        # Clean up blobs
        if os.path.exists(blobs_path):
            for item in os.listdir(blobs_path):
                item_path = os.path.join(blobs_path, item)
                if os.path.isfile(item_path):
                    os.remove(item_path)
        
        # Print progress every 10 batches
        if (j + 1) % 10 == 0:
            print(f"  Layer {i}: {j+1}/{num_batches} batches completed")
          
    sae.cpu()
    del sae
    t.cuda.empty_cache()
    gc.collect()
    print(f"✅ Completed layer {i}")

print(f"\n🎉 All files saved to: ./sae_acts_{model_ver}/shakespeare/")

/tmp/ipykernel_110/4030573705.py:14: DeprecationWarning: Indexing SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae = SAE.from_pretrained(


  Layer 7: 10/100 batches completed
  Layer 7: 20/100 batches completed
  Layer 7: 30/100 batches completed
  Layer 7: 40/100 batches completed
  Layer 7: 50/100 batches completed
  Layer 7: 60/100 batches completed
  Layer 7: 70/100 batches completed
  Layer 7: 80/100 batches completed
  Layer 7: 90/100 batches completed
  Layer 7: 100/100 batches completed
✅ Completed layer 7


layer_8/width_16k/average_l0_71/params.n(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

  Layer 8: 10/100 batches completed
  Layer 8: 20/100 batches completed
  Layer 8: 30/100 batches completed
  Layer 8: 40/100 batches completed
  Layer 8: 50/100 batches completed
  Layer 8: 60/100 batches completed
  Layer 8: 70/100 batches completed
  Layer 8: 80/100 batches completed
  Layer 8: 90/100 batches completed
  Layer 8: 100/100 batches completed
✅ Completed layer 8


layer_9/width_16k/average_l0_73/params.n(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

...

  Layer 23: 10/100 batches completed
  Layer 23: 20/100 batches completed
  Layer 23: 30/100 batches completed
  Layer 23: 40/100 batches completed
  Layer 23: 50/100 batches completed
  Layer 23: 60/100 batches completed
  Layer 23: 70/100 batches completed
  Layer 23: 80/100 batches completed
  Layer 23: 90/100 batches completed
  Layer 23: 100/100 batches completed
✅ Completed layer 23


layer_24/width_16k/average_l0_73/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

  Layer 24: 10/100 batches completed
  Layer 24: 20/100 batches completed
  Layer 24: 30/100 batches completed
  Layer 24: 40/100 batches completed
  Layer 24: 50/100 batches completed
  Layer 24: 60/100 batches completed
  Layer 24: 70/100 batches completed
  Layer 24: 80/100 batches completed
  Layer 24: 90/100 batches completed
  Layer 24: 100/100 batches completed
✅ Completed layer 24


layer_25/width_16k/average_l0_116/params(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

  Layer 25: 10/100 batches completed
  Layer 25: 20/100 batches completed
  Layer 25: 30/100 batches completed
  Layer 25: 40/100 batches completed
  Layer 25: 50/100 batches completed
  Layer 25: 60/100 batches completed
  Layer 25: 70/100 batches completed
  Layer 25: 80/100 batches completed
  Layer 25: 90/100 batches completed
  Layer 25: 100/100 batches completed
✅ Completed layer 25

🎉 All files saved to: ./sae_acts_2b/shakespeare/


In [14]:
import torch

# Path to your activations file
path = "./sae_acts_2b/shakespeare/modern_layer0_idx10.pt"

# Load the tensor
activations = torch.load(path)

print(type(activations))
print(activations.shape)


<class 'torch.Tensor'>
torch.Size([10, 15, 16384])
